# 🏁 Dataset Finalization — Phase 4.7

Apply locked panel decisions, finalize image readiness, filter annotations after SAM QA, and save the inputs for Phase 5 export.

Selected policy: **B1/C images with invalid panel masks are excluded from the final training dataset.**


## 1️⃣ Setup + Load Inputs


In [1]:
!pip install -q pandas matplotlib tqdm


In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd

DRIVE_BASE = Path('/content/drive/MyDrive/ai builders/dataset/cleaned_v3')
SAM_OUT = DRIVE_BASE / 'sam_outputs'

MANIFEST_PATH = DRIVE_BASE / 'final_manifest.csv'
FINAL_ANNOTATIONS_V2_PATH = DRIVE_BASE / 'final_annotations_v2.csv'
PANEL_OUTLIERS_V2_PATH = SAM_OUT / 'outlier_panel_v2.csv'
PROGRESS_PATH = SAM_OUT / 'sam_progress.csv'

FINAL_MANIFEST_V2_PATH = DRIVE_BASE / 'final_manifest_v2.csv'
TRAINING_ANNOTATIONS_FINAL_PATH = DRIVE_BASE / 'training_annotations_final.csv'
PANEL_DECIDED_PATH = SAM_OUT / 'outlier_panel_decided.csv'
FINALIZATION_SUMMARY_PATH = SAM_OUT / 'dataset_finalization_summary.json'

for path in [MANIFEST_PATH, FINAL_ANNOTATIONS_V2_PATH, PANEL_OUTLIERS_V2_PATH, PROGRESS_PATH]:
    if not path.exists():
        raise FileNotFoundError(f'Missing required file: {path}')

manifest = pd.read_csv(MANIFEST_PATH, low_memory=False)
final_annotations = pd.read_csv(FINAL_ANNOTATIONS_V2_PATH, low_memory=False)
panel_outliers = pd.read_csv(PANEL_OUTLIERS_V2_PATH, low_memory=False)
sam_progress = pd.read_csv(PROGRESS_PATH, low_memory=False)

manifest['image_id'] = manifest['image_id'].astype(str)
final_annotations['image_id'] = final_annotations['image_id'].astype(str)
panel_outliers['image_id'] = panel_outliers['image_id'].astype(str)
sam_progress['image_id'] = sam_progress['image_id'].astype(str)

print(f'✅ manifest rows:          {len(manifest):,}')
print(f'✅ final_annotations_v2:   {len(final_annotations):,}')
print(f'✅ panel_outliers_v2:      {len(panel_outliers):,}')
print(f'✅ sam_progress rows:      {len(sam_progress):,}')


Mounted at /content/drive
✅ manifest rows:          3,246
✅ final_annotations_v2:   26,705
✅ panel_outliers_v2:      127
✅ sam_progress rows:      6,205


## 2️⃣ Helpers + Locked Panel Decisions


In [3]:
PANEL_DECISIONS = {
    'ok': 'keep',
    'too_large': 'keep',
    'too_small': 'exclude',
    'noise': 'exclude',
}

EDA_BASELINE = {
    'panel_clean': 12399,
    'panel_defective': 3766,
    'dust': 6635,
    'bird_drop': 6316,
    'physical_damage': 3255,
    'leaf': 2666,
}
EDA_TOTAL = 35037

def parse_bool_series(series):
    if series.dtype == bool:
        return series
    return series.astype(str).str.strip().str.lower().isin(['true', '1', 'yes', 'y'])

def now_iso():
    return datetime.now(timezone.utc).isoformat()

def filter_panel_done(progress_df):
    if 'phase' in progress_df.columns:
        mask = (progress_df['phase'].astype(str) == 'panel') & (progress_df['status'].astype(str) == 'done')
    elif 'task' in progress_df.columns:
        mask = (progress_df['task'].astype(str) == 'panel') & (progress_df['status'].astype(str) == 'done')
    else:
        raise ValueError('sam_progress must contain either phase or task column')
    return progress_df[mask].copy()

for col in ['kept_in_dataset']:
    if col in manifest.columns:
        manifest[col] = parse_bool_series(manifest[col])
for col in ['kept_for_training', 'sam_excluded']:
    if col in final_annotations.columns:
        final_annotations[col] = parse_bool_series(final_annotations[col])

print('✅ decisions + helpers ready')


✅ decisions + helpers ready


## 3️⃣ Apply Panel Decisions


In [4]:
panel_done = filter_panel_done(sam_progress)
all_panel_image_ids = set(panel_done['image_id'].astype(str))

panel_outliers['decision'] = panel_outliers['flag'].map(PANEL_DECISIONS)
if panel_outliers['decision'].isna().any():
    unknown_flags = sorted(panel_outliers.loc[panel_outliers['decision'].isna(), 'flag'].dropna().unique())
    raise ValueError(f'Unknown panel flags without decision mapping: {unknown_flags}')

outlier_ids = set(panel_outliers['image_id'].astype(str))
ok_ids = all_panel_image_ids - outlier_ids
kept_panel_ids = set(panel_outliers.loc[panel_outliers['decision'] == 'keep', 'image_id'].astype(str))
excluded_panel_ids = set(panel_outliers.loc[panel_outliers['decision'] == 'exclude', 'image_id'].astype(str))
final_keep_panel = ok_ids | kept_panel_ids

print(f'Panel masks in progress: {len(all_panel_image_ids):,}')
print(f'Panel outlier rows:      {len(panel_outliers):,}')
print(f'Panel ok ids inferred:   {len(ok_ids):,}')
print(f'Panel keep:              {len(final_keep_panel):,}')
print(f'Panel exclude:           {len(excluded_panel_ids):,}')
print(f'Total decisions:         {len(final_keep_panel) + len(excluded_panel_ids):,}')

if len(final_keep_panel) != 901 or len(excluded_panel_ids) != 107:
    print('⚠️ Counts differ from expected keep=901/exclude=107. Inspect panel flags before Phase 5.')
if len(final_keep_panel) + len(excluded_panel_ids) != len(all_panel_image_ids):
    raise ValueError('Panel keep + exclude does not cover all SAM panel images.')

display(panel_outliers.groupby(['flag', 'decision']).size().reset_index(name='count'))


Panel masks in progress: 1,008
Panel outlier rows:      127
Panel ok ids inferred:   881
Panel keep:              901
Panel exclude:           107
Total decisions:         1,008


,flag,decision,count
0,noise,exclude,7
1,too_large,keep,20
2,too_small,exclude,100


## 4️⃣ Update Manifest Readiness


In [5]:
def panel_valid_for_image(image_id):
    image_id = str(image_id)
    if image_id in final_keep_panel:
        return True
    if image_id in excluded_panel_ids:
        return False
    return pd.NA

manifest['panel_mask_valid'] = manifest['image_id'].apply(panel_valid_for_image)
manifest.loc[manifest['final_category'].isin(['A', 'A_partial']), 'panel_mask_valid'] = pd.NA

def is_training_ready(row):
    if not bool(row.get('kept_in_dataset', False)):
        return False
    category = str(row.get('final_category'))
    if category in ['A', 'A_partial']:
        return True
    if category in ['B1', 'C']:
        return bool(row.get('panel_mask_valid') == True)
    return False

manifest['training_ready'] = manifest.apply(is_training_ready, axis=1)

print('panel_mask_valid distribution:')
display(manifest['panel_mask_valid'].value_counts(dropna=False).rename_axis('panel_mask_valid').reset_index(name='images'))

print('training_ready distribution:')
display(manifest['training_ready'].value_counts(dropna=False).rename_axis('training_ready').reset_index(name='images'))

print('Training-ready by category:')
display(manifest.loc[manifest['training_ready'], 'final_category'].value_counts().rename_axis('final_category').reset_index(name='images'))



panel_mask_valid distribution:


,panel_mask_valid,images
0,<NA>,2238
1,True,901
2,False,107


training_ready distribution:


,training_ready,images
0,True,2343
1,False,903


Training-ready by category:


,final_category,images
0,A,1310
1,C,752
2,B1,149
3,A_partial,132


## 5️⃣ Build Final Training Annotations


In [6]:
ready_ids = set(manifest.loc[manifest['training_ready'], 'image_id'].astype(str))

if 'kept_for_training' not in final_annotations.columns:
    raise ValueError('final_annotations_v2.csv must contain kept_for_training from defect decision phase.')

training_annotations = final_annotations[
    final_annotations['image_id'].astype(str).isin(ready_ids)
    & final_annotations['kept_for_training'].astype(bool)
].copy()

print(f'Final training-ready images: {len(ready_ids):,}')
print(f'Final training annotations:  {len(training_annotations):,}')
print('Per class:')
display(training_annotations['class_unified'].value_counts().rename_axis('class_unified').reset_index(name='annotations'))


Final training-ready images: 2,343
Final training annotations:  26,347
Per class:


,class_unified,annotations
0,panel_clean,10244
1,bird_drop,5299
2,panel_defective,2830
3,dust,2749
4,leaf,2654
5,physical_damage,2571


## 6️⃣ Annotation Counts vs EDA Baseline


In [7]:
final_counts = training_annotations['class_unified'].value_counts().to_dict()
rows = []
for cls, eda_count in EDA_BASELINE.items():
    final_count = int(final_counts.get(cls, 0))
    drop = eda_count - final_count
    drop_pct = (drop / eda_count * 100.0) if eda_count else 0.0
    rows.append({
        'Class': cls,
        'Annotations': final_count,
        'EDA baseline': eda_count,
        'Dropped': drop,
        'Drop %': round(drop_pct, 2),
    })
rows.append({
    'Class': 'TOTAL',
    'Annotations': int(len(training_annotations)),
    'EDA baseline': EDA_TOTAL,
    'Dropped': int(EDA_TOTAL - len(training_annotations)),
    'Drop %': round((EDA_TOTAL - len(training_annotations)) / EDA_TOTAL * 100.0, 2),
})
comparison_table = pd.DataFrame(rows)
display(comparison_table)


,Class,Annotations,EDA baseline,Dropped,Drop %
0,panel_clean,10244,12399,2155,17.38
1,panel_defective,2830,3766,936,24.85
2,dust,2749,6635,3886,58.57
3,bird_drop,5299,6316,1017,16.10
4,physical_damage,2571,3255,684,21.01
5,leaf,2654,2666,12,0.45
6,TOTAL,26347,35037,8690,24.80


## 7️⃣ Save Final Files


In [8]:
manifest.to_csv(FINAL_MANIFEST_V2_PATH, index=False)
training_annotations.to_csv(TRAINING_ANNOTATIONS_FINAL_PATH, index=False)
panel_outliers.to_csv(PANEL_DECIDED_PATH, index=False)
comparison_table.to_csv(SAM_OUT / 'final_annotation_counts_vs_eda.csv', index=False)

print(f'✅ Saved: {FINAL_MANIFEST_V2_PATH} ({len(manifest):,} rows)')
print(f'✅ Saved: {TRAINING_ANNOTATIONS_FINAL_PATH} ({len(training_annotations):,} rows)')
print(f'✅ Saved: {PANEL_DECIDED_PATH} ({len(panel_outliers):,} rows)')
print(f'✅ Saved: {SAM_OUT / "final_annotation_counts_vs_eda.csv"}')


✅ Saved: /content/drive/MyDrive/ai builders/dataset/cleaned_v3/final_manifest_v2.csv (3,246 rows)
✅ Saved: /content/drive/MyDrive/ai builders/dataset/cleaned_v3/training_annotations_final.csv (26,347 rows)
✅ Saved: /content/drive/MyDrive/ai builders/dataset/cleaned_v3/sam_outputs/outlier_panel_decided.csv (127 rows)
✅ Saved: /content/drive/MyDrive/ai builders/dataset/cleaned_v3/sam_outputs/final_annotation_counts_vs_eda.csv


## 8️⃣ Final Summary


In [9]:
ready = manifest['training_ready'].astype(bool)
cats = manifest.loc[ready, 'final_category'].value_counts()
summary = {
    'phase': 'dataset finalization',
    'created_at': now_iso(),
    'panel_masks_total': int(len(all_panel_image_ids)),
    'panel_masks_kept': int(len(final_keep_panel)),
    'panel_masks_excluded': int(len(excluded_panel_ids)),
    'training_ready_images': int(ready.sum()),
    'training_ready_by_category': {str(k): int(v) for k, v in cats.to_dict().items()},
    'training_annotations': int(len(training_annotations)),
    'class_counts': {str(k): int(v) for k, v in training_annotations['class_unified'].value_counts().to_dict().items()},
    'outputs': {
        'final_manifest_v2': str(FINAL_MANIFEST_V2_PATH),
        'training_annotations_final': str(TRAINING_ANNOTATIONS_FINAL_PATH),
        'outlier_panel_decided': str(PANEL_DECIDED_PATH),
        'annotation_counts_vs_eda': str(SAM_OUT / 'final_annotation_counts_vs_eda.csv'),
    },
    'ready_for_phase_5': True,
}
with open(FINALIZATION_SUMMARY_PATH, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print('╔' + '═' * 74 + '╗')
print('║  Dataset Finalization — Ready for Phase 5'.ljust(75) + '║')
print('╠' + '═' * 74 + '╣')
print(f'║  PANEL DECISIONS APPLIED:                                      ║')
print(f'║    Kept panel masks:      {len(final_keep_panel):>5}  (ok + too_large)          ║')
print(f'║    Excluded panel masks:  {len(excluded_panel_ids):>5}  (too_small + noise)      ║')
print('╠' + '═' * 74 + '╣')
print(f'║  FINAL TRAINING DATASET:                                      ║')
print(f'║    Total images:          {int(ready.sum()):>5}                           ║')
print(f'║      Category A:          {cats.get("A", 0):>5}                           ║')
print(f'║      Category A_partial:  {cats.get("A_partial", 0):>5}                           ║')
print(f'║      Category B1:         {cats.get("B1", 0):>5}  (valid SAM panel)       ║')
print(f'║      Category C:          {cats.get("C", 0):>5}  (valid SAM masks)       ║')
print(f'║    Total annotations:     {len(training_annotations):>6,}                         ║')
print('╠' + '═' * 74 + '╣')
print(f'║  Files saved to Drive:                                        ║')
print(f'║    final_manifest_v2.csv                                      ║')
print(f'║    training_annotations_final.csv                             ║')
print(f'║    sam_outputs/outlier_panel_decided.csv                      ║')
print(f'║    sam_outputs/dataset_finalization_summary.json              ║')
print('╠' + '═' * 74 + '╣')
print(f'║  Ready for Phase 5: Merge + YOLO Export                       ║')
print('╚' + '═' * 74 + '╝')
print(f'💾 Saved summary: {FINALIZATION_SUMMARY_PATH}')


╔══════════════════════════════════════════════════════════════════════════╗
║  Dataset Finalization — Ready for Phase 5                                ║
╠══════════════════════════════════════════════════════════════════════════╣
║  PANEL DECISIONS APPLIED:                                      ║
║    Kept panel masks:        901  (ok + too_large)          ║
║    Excluded panel masks:    107  (too_small + noise)      ║
╠══════════════════════════════════════════════════════════════════════════╣
║  FINAL TRAINING DATASET:                                      ║
║    Total images:           2343                           ║
║      Category A:           1310                           ║
║      Category A_partial:    132                           ║
║      Category B1:           149  (valid SAM panel)       ║
║      Category C:            752  (valid SAM masks)       ║
║    Total annotations:     26,347                         ║
╠════════════════════════════════════════════════════════════════